# **SIZE RECOMMENDTIONS**

#### **IMPORTS**

In [ ]:
# Utilities.

import zipfile
import re

# Data Processing.

import pandas as pd
import numpy as np

# Visualization.

import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Save/Load Models
import joblib 

#### **COLLECTING DATA**

In [ ]:
zip_path = "data/archive (1).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

print("Extracted!")

In [ ]:
data_1 = pd.read_json("data/renttherunway_final_data.json", lines= True)

In [ ]:
print("data1 shape: ",data_1.shape)

In [ ]:
data_1.info()

In [ ]:
data_1.columns.tolist()

In [ ]:
print(data_1.isnull().sum())

In [ ]:
data_1['item_id'].duplicated().sum()

#### **COPYING DATA**

In [ ]:
df = data_1.copy()

#### **ANALYSING AND CORRECTING DATA**

In [ ]:
# see values in body type.

df['body type'].value_counts()

In [ ]:
# dropping rows.

df = df.dropna(subset= ['body type', 'size', 'height', 'weight', 'age'])

In [ ]:
# cleaning height.

def parse_height(h):
    if pd.isna(h):
        return np.nan

    match = re.match(r"(\d+)'\s*(\d+)\"", str(h))
    if match:
        feet = int(match.group(1))
        inches = int(match.group(2))
        return (feet * 12) + inches
    return np.nan

df['height_inches'] = df['height'].apply(parse_height)

In [ ]:
# cleaning weight

def parse_weight(w):
    if pd.isna(w):
        return np.nan
    try:

        return float(str(w).replace('lbs', '').strip())
    except:
        return np.nan

df['weight_lbs'] = df['weight'].apply(parse_weight)

In [ ]:
# Map the body types

body_type_map = {
    'petite': 'Petite',
    'hourglass': 'Average',
    'athletic': 'Average',
    'pear': 'Broad',
    'apple': 'Broad',
    'full_bust': 'Broad',
    'straight & narrow': 'Average'
}

df['mapped_body_type'] = df['body type'].map(body_type_map)

In [ ]:
# dropping rows where the height and weight parsing failed.

df = df.dropna(subset=['height_inches', 'weight_lbs', 'mapped_body_type'])

In [ ]:
def categorize_size (size):
    if size <= 6:
        return 'Small'
    elif size <= 12:
        return 'Medium'
    else:
        return 'Large'

df['size_category'] = df['size'].apply(categorize_size)

print("New Target Destribution: ")
print(df['size_category'].value_counts())

In [ ]:
df.info()

In [ ]:
print("Data Cleaned! Final Shape: ", df.shape)
print("\nSample of clean data: ")
print(df[['age', 'height_inches', 'weight_lbs', 'mapped_body_type', 'size', 'fit']].head(11))

#### **CORRELATION ANALYSIS**

In [ ]:
# Heatmap (Correlation matrix).

numeric_cols = ['age', 'height_inches', 'weight_lbs', 'size']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap= 'coolwarm', fmt=".2f", center=0)
plt.title('Correlation Matrix: Age, Height, Weight, and Size')
plt.show()

In [ ]:
# Average size by body type.

print("Average Size by body type.")
print(df.groupby('mapped_body_type')['size'].mean().sort_values(ascending= False).round(2))

#### **MODEL**

In [ ]:
# 1. Select features and targets.

print("Preparing data for the model...")

features = ['age', 'height_inches', 'weight_lbs', 'mapped_body_type']
target = 'size_category'

model_data = df.dropna(subset=['age', 'size_category']).copy()
X = model_data[features]
y = model_data[target]

print("X shape: ", X.shape)
print("y shape: ", y.shape)

In [ ]:
# 2. One hot encoding.
X = pd.get_dummies(X, columns=['mapped_body_type'], drop_first= True)

In [ ]:
# 3. Train/Test Split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state= 42
)

In [ ]:
# 4. Train.
print("Training Random Forest...")
rf_size_model = RandomForestClassifier(
    n_estimators= 100,
    random_state= 42,
    n_jobs= -1
)
rf_size_model.fit(X_train, y_train)

In [ ]:
# 5. Evaluate
preds = rf_size_model.predict(X_test)
acc = accuracy_score(y_test, preds)

In [ ]:
print(f"\nExact Size Accuracy: {acc * 100:.2f}%")
print("\n--- Classification Report ---")
print(classification_report(y_test, preds, zero_division=0))

#### **SAVING MODEL**

In [ ]:
joblib.dump(rf_size_model, 'models/size_category_model.pkl')

model_columns = X_train.columns.tolist()
joblib.dump(model_columns, 'models/model_columns.pkl')

print("All Saved")

### **PREDICTING MODEL**

In [ ]:
def get_size_recommendation(age, height_inches, weight_lbs, mapped_body_type):

    input_data= pd.DataFrame([{
        'age': age,
        'height_inches': height_inches,
        'weight_lbs': weight_lbs,
        'mapped_body_type': mapped_body_type
    }])

    input_data = pd.get_dummies(input_data, columns=['mapped_body_type'], drop_first= True)

    input_data = input_data.reindex(columns=X_train.columns, fill_value= 0)

    prediction = rf_size_model.predict(input_data)[0]

    return prediction

In [ ]:
print("Test 1 (Average 5'6\", 140lbs):")
print(get_size_recommendation(age= 28, height_inches= 66, weight_lbs= 140, mapped_body_type= 'Average'))
print()

print("Test 2 (Tall-Heavy 5'9\", 180lbs):")
print(get_size_recommendation(age= 35, height_inches= 69, weight_lbs= 180, mapped_body_type= 'Tall-Heavy'))
print()

print("Test 3 (Petite 5'2\", 110lbs):")
print(get_size_recommendation(age= 22, height_inches= 62, weight_lbs= 110, mapped_body_type= 'Petite'))
print()